# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load config
from IPython.display import display
import sys
import os
from pathlib import Path


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

from model_in_the_loop.utils.hydra_utils import load_config,set_env_vars
cfg = load_config()
set_env_vars(cfg)  # set env variables for repo and data paths
print(f"is_ensemble_model: {cfg.model_configs.is_ensemble_model}, only_train_readout: {cfg.model_configs.only_train_readout}")

In [3]:
from model_in_the_loop.core.dj_wrappers import (DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper)

from model_in_the_loop.utils.file_management import copy_rec_files,create_directory_structure,rm_all_experiment_dirs,clear_data_dump_dir
from model_in_the_loop.utils.transform_to_avi_stimulus import create_single_mei_avis_and_metadata,create_rf_test_dir
from model_in_the_loop.utils.simple_logging import log
from model_in_the_loop.utils.plotting import show_all_rois_plot

### Create processing components (connect them to DB)

In [4]:
# create preprocessor
dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore

                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [5]:
dj_table_holder.setup()


In [ ]:
# dj_table_holder.clear_tables("all",safemode=False)
# rm_all_experiment_dirs(cfg.DJ.userinfo.data_dir)

In [6]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    cfg=cfg,
    seeds=[111,222]
)

## During the experimet

### Move files from server to the repo 

In [ ]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,)

copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

### Preprocessing

In [ ]:
preprocessor.upload_iteration_metadata()

In [7]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
    print(f"Missing field key found: {field_key}")
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

In [ ]:
# compute 
preprocessor.add_iteration_roi_mask(field_key=field_key)
preprocessor.add_iteration_rois()
preprocessor.add_iteration_traces()

### qualty and RF

In [8]:
quality_type_analysis_wrapper.compute_analysis(
    field_key=field_key)

# filter 
passed_roi_ids_chirp_mb = quality_type_analysis_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
    d_qi_min =cfg.quality_filtering["d_qi_min"],
    qidx_min=cfg.quality_filtering["chirp_qi_min"],
    celltypes=cfg.quality_filtering["celltypes"],
    classifier_confidence= 0), # NOTE: this cfg.quality_filtering["classifier_confidence"])
if len(passed_roi_ids_chirp_mb) == 0:
    raise ValueError("No ROIs passed the quality criterion for quality and type.")
print(f"{len(passed_roi_ids_chirp_mb)} ROIs passed quality chirp mb filtering: {passed_roi_ids_chirp_mb}")



In [9]:
# sta 
sta_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_chirp_mb,)


In [10]:
assert ((dj_table_holder("STA")() & field_key).fetch("roi_id") == passed_roi_ids_chirp_mb).all(), "STA roi_id does not match passed roi_ids from quality and type filtering."

In [11]:
# filter
passed_roi_ids_sta = sta_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
                                                               rf_qidx_min=0)# cfg.quality_filtering["rf_qidx_min"])
if len(passed_roi_ids_sta) == 0:
    raise ValueError("No ROIs passed the quality criterion for STA.")
print(f"{len(passed_roi_ids_sta)} ROIs passed STA filtering with rf_qidx_min={cfg.quality_filtering["rf_qidx_min"]}: {passed_roi_ids_sta}")

if len(passed_roi_ids_sta) < 25:
    raise ValueError(f"Less than 25 ROIs passed the quality criterion for STA. Found {len(passed_roi_ids_sta)} ROIs. Consider lowering the rf_qidx_min threshold.")


In [ ]:
if len(passed_roi_ids_sta) >= 25:
    print("MORE THAN 25 ROIS, selecting 25 best")
    top_n_rois_sta,top_n_scores = sta_wrapper.list_top_n_rois_by_qidx(field_key=field_key,
                                                    n=25,)
    passed_roi_ids_sta = top_n_rois_sta
    print(top_n_rois_sta)
    print(top_n_scores)
    print(f"Using top 25 rois based on rf_qidx: {passed_roi_ids_sta}")


# if len(passed_roi_ids_sta) >= 25:
#     print("MORE THAN 25 ROIS, selecting 25 best")
#     top_n_rois_sta,top_n_scores = sta_wrapper.list_top_n_rois_by_qidx(field_key=field_key,
#                                                     n=35,)
#     passed_roi_ids_sta = top_n_rois_sta
#     print(top_n_rois_sta)
#     print(top_n_scores)
#     print(f"Using top 25 rois based on rf_qidx: {passed_roi_ids_sta}")

### MEI

In [14]:
# compute
random_seed_mei_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_sta,
    )

In [ ]:
random_seed_mei_wrapper.mei_subanalysis()

In [ ]:
### to adjust testset correl threshold
# random_seed_mei_wrapper.neuron_testset_correls
# # random_seed_mei_wrapper.quality_filtering["min_testset_correl"] = 0
# # random_seed_mei_wrapper.apply_quality_filter()
# # random_seed_mei_wrapper.mei_subanalysis()

## visualize with GUI

In [ ]:
random_seed_mei_wrapper.load_all_data_from_dir(
    load_dir = "/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/mei2/GCL2_20251008_213910"
)

In [ ]:
from model_in_the_loop.core.gui import ExtendedAutoRoiGui
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers= [quality_type_analysis_wrapper,sta_wrapper,random_seed_mei_wrapper],
    # JUST VIS
    do_not_compute_only_visualize = True,
    
    field_key=field_key,
    canvas_width=30)

In [17]:
display(gui.start_gui())

### RF

In [ ]:
f,a = show_all_rois_plot(
    dj_table_holder=dj_table_holder,
    wrapper=sta_wrapper,
    field_key=field_key
    )

### MEI

In [ ]:
f,a = show_all_rois_plot(
    dj_table_holder=dj_table_holder,
    wrapper=random_seed_mei_wrapper,
    field_key=field_key
    )

# The stimulus

## select roi ids

In [ ]:
throw_out_these_rois = []

In [ ]:
only_these_rois = [45,77,82,] + [52,51,2]

In [ ]:
if only_these_rois != []:
    roi_ids_list = only_these_rois
    print(f"Using only these rois: {roi_ids_list}")
else:
    roi_ids_list = [roi for roi in passed_roi_ids_sta if roi not in throw_out_these_rois]
    print(f"Final roi ids list length: {len(roi_ids_list)}")

### RFs

In [ ]:
create_rf_test_dir(
    roi_ids_list= [40,45,62,77,82],
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table=dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=cfg.paths.stimulus_output_dir,
    field_restriction= field_key,
)

In [ ]:
roi_id2mei_ids,roi_id2mei_info = random_seed_mei_wrapper.select_subset_of_meis_for_each_roi(
    only_consider_these_rois=roi_ids_list,
    neuron_data_dict = random_seed_mei_wrapper.neuron_data_dict,
    new_session_id = random_seed_mei_wrapper.new_session_id,
    mei_data_container = random_seed_mei_wrapper.mei_data_container,
    readout_idx_wmei2rois = random_seed_mei_wrapper.readout_idx_wmei2rois,
    n_stimuli_total = 6,
    )
print(roi_id2mei_ids)

### meis

In [ ]:
create_single_mei_avis_and_metadata(
    rois_selected=roi_ids_list,
    roi_id2mei_ids = roi_id2mei_ids,    
    mei_data_container=random_seed_mei_wrapper.mei_data_container,
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table= dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=cfg.paths.stimulus_output_dir,
    field_restriction=field_key,
)



# Save data

### RFs

In [ ]:
import datetime
now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_rf_dir = os.path.join(cfg.paths.repo_directory,"model_in_the_loop/data/online_computed_data",f"rf_test_stimuli_{now}")
os.makedirs(save_rf_dir,exist_ok=True)

create_rf_test_dir(
    roi_ids_list= roi_ids_list,
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table=dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=save_rf_dir
)

### meis

In [ ]:

random_seed_mei_wrapper.save_all_data_to_dir(save_dir_parent=random_seed_mei_wrapper.save_dir_parent)
# save data 
import datetime
now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_mei_dir = os.path.join(cfg.paths.repo_directory,"model_in_the_loop/data/online_computed_data",f"mei_test_stimuli_{now}")
os.makedirs(save_mei_dir,exist_ok=True)



# Clean up

In [ ]:
userinput = input("Cleanup? (y/n): ")
if userinput.lower() == 'y':
    dj_table_holder.clear_tables("all")


# shit goes south